# 🩺 TriageCore - Faz 2: Google Colab T4 GPU Fine-Tuning (QLoRA & PEFT)

Bu notebook, **TriageCore** projesinin 2. Fazı için hazırlanmıştır. 
Tamamen açık ağırlıklı **Meta Llama-3-8B-Instruct** modelini, Google Colab'in ücretsiz **Nvidia T4 GPU (15.3 GB VRAM)** ortamında bellek sınırlarına takılmadan eğitmek üzere **4-bit QLoRA (Quantized Low-Rank Adaptation)** ve **PEFT** tekniklerini kullanır.

---

### 1. Adım: GPU Donanımını ve VRAM Durumunu Kontrol Etme

In [ ]:
# Colab uzerinde atanan GPU modelini ve toplam VRAM kapasitesini konsola yazdirir
!nvidia-smi

### 2. Adım: Gerekli Açık Kaynak Kütüphanelerin Kurulumu

* `transformers`: Model mimarisi ve ağırlıkları yönetir.
* `peft`: LoRA adaptörlerini modele entegre eder.
* `bitsandbytes`: 4-bit normal float (NF4) kuantizasyon motorunu sağlar.
* `trl`: Supervised Fine-Tuning (SFTTrainer) pipeline'ını sağlar.
* `accelerate`: Çoklu donanım ve bellek optimizasyonlarını yürütür.

In [ ]:
# Transformers kutuphanesinin guncel surumunu sessizce kurar
!pip install -q transformers>=4.40.0

# LoRA gibi hafif egitim adaptorlerini yoneten PEFT kutuphanesini kurar
!pip install -q peft>=0.10.0

# 4-bit ve 8-bit kuantizasyon motoru bitsandbytes'i kurar
!pip install -q bitsandbytes>=0.43.0

# SFTTrainer ve veri hazirlama pipeline'i sunan TRL kutuphanesini kurar
!pip install -q trl>=0.8.6

# GPU bellek optimizasyon araci accelerate'i kurar
!pip install -q accelerate>=0.29.0

# HuggingFace veri kumelerini yoneten datasets kutuphanesini kurar
!pip install -q datasets>=2.18.0

### 3. Adım: PyTorch ve Bellek Ayarlarının İçe Aktarılması

In [ ]:
# Isletim sistemi islemleri icin os kutuphanesini ice aktarir
import os

# PyTorch derin ogrenme kutuphanesini ice aktarir
import torch

# HuggingFace uzerinden veri seti yukleme fonksiyonunu ice aktarir
from datasets import load_dataset

# PEFT bilesenlerini ice aktarir
from peft import (
    LoraConfig,  # LoRA hiperparametre konfigurasyon sinifi
    get_peft_model,  # Modele LoRA adaptorlerini ekleyen fonksiyon
    prepare_model_for_kbit_training,  # Modeli 4-bit gradyan egitimine hazirlayan fonksiyon
)

# Transformers bilesenlerini ice aktarir
from transformers import (
    AutoModelForCausalLM,  # Metin uretim modelini yukleyen sinif
    AutoTokenizer,  # Modele ait tokenizer'i yukleyen sinif
    BitsAndBytesConfig,  # 4-bit kuantizasyon konfigurasyon sinifi
    TrainingArguments,  # Egitim dongusu argumanlari sinifi
)

# TRL kutuphanesinden Supervised Fine-Tuning egiticisini ice aktarir
from trl import SFTTrainer

# Kullanilan CUDA cihazinin adini konsola yazdirir
print(f"CUDA Baglantisi Basarili: {torch.cuda.get_device_name(0)}")

### 4. Adım: 4-Bit NormalFloat (NF4) Kuantizasyon Konfigürasyonu

Bu adım, 8 milyar parametrelik modeli 16 GB'den yaklaşık **4 GB VRAM**'e indirir.

In [ ]:
# BitsAndBytes 4-bit yapilandirma nesnesi olusturuluyor
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                     # Modeli 4-bit kuantize sekilde VRAM'e yukler
    bnb_4bit_quant_type="nf4",             # Bilgi kaybi en az olan NormalFloat4 formatini secer
    bnb_4bit_use_double_quant=True,         # Kuantizasyon sabitlerini de kuantize ederek ek bellek kazanir
    bnb_4bit_compute_dtype=torch.float16,  # T4 GPU mimarisinde matris islemlerini float16 hassasiyetinde yapar
)

# Yapilandirma onay mesajini ekrana basar
print("BitsAndBytes 4-bit NF4 Konfigurasyonu Hazirlandi.")

### 5. Adım: Model ve Tokenizer'ın Yüklenmesi

In [ ]:
# Egitilecek olan acik agirlikli temel model kimligi
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

# Hizli Rust tabanli Tokenizer VRAM disinda sistem RAM'ine yuklenir
tokenizer = AutoTokenizer.from_pretrained(
    model_id,               # Model kimligi
    use_fast=True,          # C++ / Rust hizlandirmali tokenizer kullan
    trust_remote_code=True, # Ozel kod varsa calistirmaya izin ver
)

# Eger pad token tanimli degilse, sonlandirma token'ina esitlenir
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Causal Language Modeling icin padding'in sag tarafta olmasi sarttir
tokenizer.padding_side = "right"

# 4-bit kuantize base model VRAM'e yuklenir (~4.2 GB yer kaplar)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config, # 4-bit kuralini uygula
    device_map="auto",              # Katmanlari GPU'ya en optimize sekilde dagit
    trust_remote_code=True,
)

# Modeli k-bit egitim icin hazirla (LayerNorm katmanlarini float32 yaparak numerik stabilite saglar)
model = prepare_model_for_kbit_training(model)

# Aktivasyon bellek tuketimini %60 azaltan gradient checkpointing'i ac
model.gradient_checkpointing_enable()

# Yukleme basari mesajini yazdir
print("Model ve Tokenizer basariyla VRAM'e yuklendi!")

### 6. Adım: LoRA (Low-Rank Adaptation) Katmanlarının Entegrasyonu

In [ ]:
# LoRA konfigurasyon nesnesi tanimlaniyor
lora_config = LoraConfig(
    r=16,                                    # Adaptor rank boyutu (16 klinik triyaj icin mukemmel dengedir)
    lora_alpha=32,                           # Olcekleme parametresi (alpha/r = 2 carpani uygulanir)
    target_modules=[                         # LoRA eklenecek tum lineer projeksiyon matrisleri
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,                       # Ezberlemeyi onleyici %5 dropout orani
    bias="none",                             # Ekstra bias ogrenilmez
    task_type="CAUSAL_LM",                   # Metin uretim gorevi
)

# Donuk base modele LoRA katmanlari eklenir
model = get_peft_model(model, lora_config)

# Egitilecek olan kucuk parametre oranini konsola basar (Genellikle %0.1 - %0.3)
model.print_trainable_parameters()

### 7. Adım: Faz 1'de Ürettiğimiz Veri Setinin Yüklenmesi

In [ ]:
# Egitim ve dogrulama verilerinin yollari tanimlaniyor
# Not: Dosyalari sol paneldeki Dosyalar (Files) kismina yukleyebilirsiniz
data_files = {
    "train": "train.jsonl",
    "validation": "val.jsonl",
}

# Eger dosyalar henuz yuklenmediyse ornek veri olusturur
if not os.path.exists("train.jsonl"):
    print("[!] train.jsonl bulunamadi. Projedeki data/sample_triage.jsonl dosyasini yukleyiniz.")
else:
    # JSONL dosyalari HuggingFace dataset nesnesine donusturulur
    dataset = load_dataset("json", data_files=data_files)
    # Yuklenen ornek sayilarini ekrana basar
    print(f"Veri Seti Hazir: {len(dataset['train'])} egitim, {len(dataset['validation'])} dogrulama ornegi.")

### 8. Adım: Eğitim Hiperparametreleri ve SFTTrainer Başlatma

In [ ]:
# Egitim dongusu hiperparametreleri belirleniyor
training_args = TrainingArguments(
    output_dir="./triagecore-checkpoints",     # Checkpoint'lerin yazilacagi dizin
    num_train_epochs=3,                       # 3 tam epoch egitim
    per_device_train_batch_size=2,            # T4 GPU'da OOM onleyici kucuk batch
    gradient_accumulation_steps=8,            # 2 * 8 = 16 etkin batch boyutu
    optim="paged_adamw_8bit",                 # Bellek asimlarini engelleyen 8-bit Paged AdamW
    learning_rate=2e-4,                        # QLoRA standart ogrenme katsayisi
    lr_scheduler_type="cosine",               # Yumusak sonlanma icin kosinus zamanlayicisi
    warmup_ratio=0.05,                        # Ilk %5 adimda kademeli isinma
    logging_steps=10,                         # Her 10 adimda kaybi (loss) konsola yazdir
    save_strategy="steps",                    # Adim bazli kaydetme
    save_steps=50,                            # Her 50 adimda checkpoint olustur
    eval_strategy="steps",                    # Adim bazli dogrulama
    eval_steps=50,                            # Her 50 adimda validation loss olc
    fp16=True,                                # 16-bit float hesaplama destegi
    max_grad_norm=0.3,                        # Gradyan kirpma (patlamalari onler)
    report_to="none",                         # Konsola loglama
)

# SFTTrainer nesnesi olusturuluyor
trainer = SFTTrainer(
    model=model,                              # LoRA modeli
    train_dataset=dataset["train"],            # Egitim verisi
    eval_dataset=dataset["validation"],       # Dogrulama verisi
    peft_config=lora_config,                  # LoRA yapilandirmasi
    max_seq_length=1024,                      # 1024 token baglam siniri
    tokenizer=tokenizer,                      # Metin parcalayici
    args=training_args,                       # Egitim argumanlari
)

# Egitimi baslat
print("Egitim baslatiliyor...")
trainer.train()

### 9. Adım: Eğitilen LoRA Adaptörünün Kaydedilmesi

In [ ]:
# Cikti dizini yolu belirleniyor
output_adapter_dir = "./triagecore-lora-adapter"

# Sadece ogrenilen LoRA adapter agirliklari diske kaydedilir (~150 MB)
trainer.model.save_pretrained(output_adapter_dir)

# Tokenizer konfigurasyonu cikti klasorune yazilir
tokenizer.save_pretrained(output_adapter_dir)

# Faz 3'te birlestirmek uzere adaptoru zip dosyasina sikistirir
!zip -r triagecore-lora-adapter.zip ./triagecore-lora-adapter

print("LoRA adaptoru basariyla kaydedildi ve zip olarak paketlendi!")

### 10. Adım: Eğitilmiş Model ile Klinik Triyaj Testi (Inference Sanity Check)

In [ ]:
# Modeli degerlendirme (inference) moduna alir
model.eval()

# Ornek bir hasta sikayeti olusturuluyor
test_messages = [
    {
        "role": "system",
        "content": "You are TriageCore, an expert AI clinical triage assistant. Evaluate patient symptoms, assess medical urgency, highlight critical red-flag warnings, explain potential etiologies, and recommend clinical next steps.",
    },
    {
        "role": "user",
        "content": "I have severe retrosternal chest pain radiating to my left shoulder and jaw for 45 minutes, with diaphoresis and nausea. What should I do?",
    },
]

# Tokenizer'in yerlesik chat sablonu kullanilarak girdi promptu formatlanir
prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)

# Metin token tensorlerine donusturulur ve GPU'ya tasinir
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Gradyan hesaplamasini kapatip modelden yeni tokenlar uretmesini isteriz
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,       # En fazla 256 token uret
        temperature=0.2,          # Dusuk sicaklik ile klinik tutarliligi artir
        do_sample=True,           # Ornekleme yap
        top_p=0.9,                # Nucleus sampling
        eos_token_id=tokenizer.eos_token_id,
    )

# Uretilen tokenlari insan tarafindan okunabilir metne donusturur
response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

# Ciktiyi ekrana basar
print("\n--- TriageCore Klinik Yaniti ---\n")
print(response)